# Klasifikasi Langit: Transfer Learning yang Lebih Stabil

Notebook ini menggantikan CNN dari nol dengan EfficientNetB0 pralatih, memperbaiki pengelolaan data, dan mengekspor artefak yang konsisten untuk API FastAPI.

**Struktur folder wajib**

```text
IMAGE CLASSIFICATION LANGIT/
├── TRAIN/
│   ├── LANGIT CERAH/
│   ├── LANGIT MENDUNG/
│   └── SUNSET/
├── VALIDATION/
│   ├── LANGIT CERAH/
│   ├── LANGIT MENDUNG/
│   └── SUNSET/
└── TEST/
    ├── LANGIT CERAH/
    ├── LANGIT MENDUNG/
    └── SUNSET/
```

Jangan menjalankan evaluasi TEST hingga pemilihan model selesai.

In [ ]:
# CELL 1 — konfigurasi dan impor
import os
import json
import random
from pathlib import Path
from collections import Counter
from hashlib import md5

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
INITIAL_EPOCHS = 20
FINE_TUNE_EPOCHS = 15

tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

# UBAH HANYA BARIS INI sesuai folder Google Drive Anda.
DATASET_PATH = Path("/content/drive/MyDrive/SEMESTER 8/IMAGE CLASSIFICATION LANGIT")

TRAIN_DIR = DATASET_PATH / "TRAIN"
VAL_DIR = DATASET_PATH / "VALIDATION"
TEST_DIR = DATASET_PATH / "TEST"

EXPORT_DIR = DATASET_PATH / "export_model"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = EXPORT_DIR / "sky_classifier_best.keras"
LABELS_PATH = EXPORT_DIR / "class_names.json"

for directory in (TRAIN_DIR, VAL_DIR, TEST_DIR):
    if not directory.exists():
        raise FileNotFoundError(f"Folder tidak ditemukan: {directory}")


In [ ]:
# CELL 2 — audit jumlah gambar per kelas dan cek struktur kelas
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

def count_images_per_class(split_dir: Path) -> dict[str, int]:
    return {
        class_dir.name: sum(
            file.suffix.lower() in IMAGE_EXTENSIONS
            for file in class_dir.rglob("*")
            if file.is_file()
        )
        for class_dir in sorted(split_dir.iterdir())
        if class_dir.is_dir()
    }

train_counts = count_images_per_class(TRAIN_DIR)
val_counts = count_images_per_class(VAL_DIR)
test_counts = count_images_per_class(TEST_DIR)

distribution = pd.DataFrame(
    {"train": train_counts, "validation": val_counts, "test": test_counts}
).fillna(0).astype(int)

display(distribution)
assert set(train_counts) == set(val_counts) == set(test_counts), (
    "Nama folder kelas di TRAIN, VALIDATION, dan TEST harus sama."
)

CLASS_NAMES = sorted(train_counts)
NUM_CLASSES = len(CLASS_NAMES)
assert NUM_CLASSES == 3, f"Notebook ini saat ini mengharapkan 3 kelas, ditemukan: {NUM_CLASSES}"

imbalance_ratio = max(train_counts.values()) / min(train_counts.values())
print(f"Class names: {CLASS_NAMES}")
print(f"Rasio kelas terbesar/terkecil di train: {imbalance_ratio:.2f}")


In [ ]:
# CELL 3 — cek duplikasi file identik antar split untuk menghindari data leakage
def file_hashes(split_dir: Path) -> dict[str, str]:
    hashes = {}
    for path in split_dir.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            digest = md5(path.read_bytes()).hexdigest()
            hashes[digest] = str(path)
    return hashes

train_hashes = file_hashes(TRAIN_DIR)
val_hashes = file_hashes(VAL_DIR)
test_hashes = file_hashes(TEST_DIR)

for split_a, hashes_a, split_b, hashes_b in [
    ("TRAIN", train_hashes, "VALIDATION", val_hashes),
    ("TRAIN", train_hashes, "TEST", test_hashes),
    ("VALIDATION", val_hashes, "TEST", test_hashes),
]:
    duplicated = set(hashes_a).intersection(hashes_b)
    print(f"Duplikasi identik {split_a} ↔ {split_b}: {len(duplicated)}")
    if duplicated:
        first_hash = next(iter(duplicated))
        print("Contoh:", hashes_a[first_hash], "<->", hashes_b[first_hash])
        raise ValueError("Hapus atau tempatkan ulang file duplikat sebelum training.")


In [ ]:
# CELL 4 — buat tf.data dataset. TEST tidak diacak.
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)


In [ ]:
# CELL 5 — tampilkan sampel, lalu periksa label secara manual
plt.figure(figsize=(12, 8))
for images, labels in train_ds.take(1):
    for index in range(min(12, len(images))):
        ax = plt.subplot(3, 4, index + 1)
        plt.imshow(images[index].numpy().astype("uint8"))
        plt.title(CLASS_NAMES[int(labels[index])])
        plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 6 — class weights hanya akan berpengaruh jika jumlah kelas tidak seimbang.
train_label_ids = np.concatenate([labels.numpy() for _, labels in train_ds.unbatch().batch(4096)])
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_label_ids,
)
class_weights = {index: float(weight) for index, weight in enumerate(weights)}
print("Class weights:", class_weights)


In [ ]:
# CELL 7 — EfficientNetB0 + augmentasi ringan dan realistis
from tensorflow.keras import layers, regularizers

augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05, fill_mode="reflect"),
        layers.RandomZoom(height_factor=(-0.10, 0.10), width_factor=(-0.10, 0.10)),
        layers.RandomContrast(0.10),
    ],
    name="data_augmentation",
)

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,),
)
base_model.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,), name="image")
x = augmentation(inputs)
# Penting: training=False menjaga BatchNormalization pada backbone pralatih tetap stabil.
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = layers.Dropout(0.35, name="head_dropout")(x)
outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    kernel_regularizer=regularizers.l2(1e-4),
    name="classifier",
)(x)

model = tf.keras.Model(inputs, outputs, name="sky_efficientnetb0")
model.summary()


In [ ]:
# CELL 8 — callbacks dan training tahap 1 (classifier head)
def make_callbacks():
    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(MODEL_PATH),
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=6,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.3,
            patience=3,
            min_lr=1e-7,
            verbose=1,
        ),
    ]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.05),
    metrics=["accuracy"],
)

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    class_weight=class_weights,
    callbacks=make_callbacks(),
    verbose=1,
)


In [ ]:
# CELL 9 — fine-tuning terukur. Buka hanya layer atas backbone dan gunakan learning rate kecil.
base_model.trainable = True

# Bekukan semua kecuali sekitar 25 layer paling atas.
for layer in base_model.layers[:-25]:
    layer.trainable = False

# BatchNormalization tidak di-fine-tune pada dataset kecil.
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.02),
    metrics=["accuracy"],
)

initial_epoch = len(history_head.history["loss"])
history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=initial_epoch,
    epochs=initial_epoch + FINE_TUNE_EPOCHS,
    class_weight=class_weights,
    callbacks=make_callbacks(),
    verbose=1,
)


In [ ]:
# CELL 10 — visualisasi training dan pemuatan checkpoint terbaik secara eksplisit
history = {}
for metric in ("loss", "accuracy", "val_loss", "val_accuracy"):
    history[metric] = history_head.history.get(metric, []) + history_finetune.history.get(metric, [])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["accuracy"], label="Train")
axes[0].plot(history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["loss"], label="Train")
axes[1].plot(history["val_loss"], label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.show()

# Jangan mengevaluasi variabel model terakhir tanpa memuat checkpoint terbaik.
best_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print(f"Checkpoint terbaik dimuat dari: {MODEL_PATH}")


In [ ]:
# CELL 11 — evaluasi final satu kali pada TEST
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=0)
probabilities = best_model.predict(test_ds, verbose=0)
y_pred = np.argmax(probabilities, axis=1)
y_true = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print()
print(classification_report(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

matrix = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
display = ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(7, 6))
display.plot(ax=ax, cmap="Blues", colorbar=False)
plt.xticks(rotation=20)
plt.title("Confusion Matrix: Test Set")
plt.show()


In [ ]:
# CELL 12 — ekspor artefak yang dipakai oleh backend FastAPI
with open(LABELS_PATH, "w", encoding="utf-8") as file:
    json.dump(CLASS_NAMES, file, ensure_ascii=False, indent=2)

print("Artefak berhasil diekspor:")
print("-", MODEL_PATH)
print("-", LABELS_PATH)

# Opsional untuk Google Colab:
# from google.colab import files
# files.download(str(MODEL_PATH))
# files.download(str(LABELS_PATH))
